# 1.8) (Exercises) Matplotlib and xarray

:::{admonition} How to use these exercises
:class: note
Attempt each from scratch in the empty code cell; the distributed hand-out omits the solutions. Import numpy as `np`, matplotlib.pyplot as `plt`, and xarray as `xr`; label every axis with its unit, and seed any randomness with `default_rng`.
:::

## Exercise 1 — a labelled line plot

Given `days = np.arange(365)` and `temp_celsius = 10 + 8 * np.sin(2 * np.pi * days / 365)`, plot temperature against day of year as a line, with axis labels (including the unit) and a title.

In [ ]:
# Your solution here

:::{admonition} Solution
:class: note dropdown
```python
import numpy as np
import matplotlib.pyplot as plt
days = np.arange(365)
temp_celsius = 10 + 8 * np.sin(2 * np.pi * days / 365)
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(days, temp_celsius)
ax.set_xlabel("day of year")
ax.set_ylabel("temperature (°C)")
ax.set_title("idealised seasonal cycle")
plt.show()
```
:::

## Exercise 2 — scatter and histogram side by side

With `rng = np.random.default_rng(0)`, `x = np.arange(100)`, and `y = 0.1 * x + rng.normal(0, 1, 100)`, make a 1×2 figure: a scatter of `y` versus `x` on the left, and a histogram of `y` on the right. Label all axes.

In [ ]:
# Your solution here

:::{admonition} Solution
:class: note dropdown
```python
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)
x = np.arange(100)
y = 0.1 * x + rng.normal(0, 1, 100)
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
axes[0].scatter(x, y, s=10)
axes[0].set_xlabel("index"); axes[0].set_ylabel("value")
axes[1].hist(y, bins=20)
axes[1].set_xlabel("value"); axes[1].set_ylabel("count")
plt.tight_layout(); plt.show()
```
:::

## Exercise 3 — a 2D field with a colorbar, saved as vector

Given

```python
lon = np.linspace(6.0, 9.0, 6)
lat = np.linspace(46.0, 47.5, 4)
field_celsius = np.random.default_rng(0).normal(5, 2, size=(4, 6))
```

draw the field with `pcolormesh` and the coordinates, add a labelled colorbar and axis labels, and save the figure as `field.svg`.

In [ ]:
# Your solution here

:::{admonition} Solution
:class: note dropdown
```python
import numpy as np
import matplotlib.pyplot as plt
lon = np.linspace(6.0, 9.0, 6)
lat = np.linspace(46.0, 47.5, 4)
field_celsius = np.random.default_rng(0).normal(5, 2, size=(4, 6))
fig, ax = plt.subplots(figsize=(5, 3))
pcm = ax.pcolormesh(lon, lat, field_celsius, shading="auto")
fig.colorbar(pcm, ax=ax, label="temperature (°C)")
ax.set_xlabel("longitude (°E)")
ax.set_ylabel("latitude (°N)")
fig.savefig("field.svg")
plt.show()
```
:::

## Exercise 4 — build a labelled DataArray

From `data = np.arange(12.0).reshape(3, 4)`, build an xarray DataArray with dimensions `("lat", "lon")`, latitude coordinates `[46.0, 46.5, 47.0]`, longitude coordinates `[6.0, 6.5, 7.0, 7.5]`, and a `units` attribute of `"degC"`. Print its dims and its units.

In [ ]:
# Your solution here

:::{admonition} Solution
:class: note dropdown
```python
import numpy as np
import xarray as xr
data = np.arange(12.0).reshape(3, 4)
da = xr.DataArray(
    data,
    dims=("lat", "lon"),
    coords={"lat": [46.0, 46.5, 47.0], "lon": [6.0, 6.5, 7.0, 7.5]},
    attrs={"units": "degC", "long_name": "temperature"},
)
print(da.dims, da.attrs["units"])
```
:::

## Exercise 5 — select and reduce

Build the small Dataset below, then print the spatial mean of the first day (by position) and the time mean at latitude 47.0 (by label).

```python
rng = np.random.default_rng(0)
time = np.arange("2024-01-01", "2024-01-11", dtype="datetime64[D]")
lat = np.array([46.0, 46.5, 47.0]); lon = np.array([6.0, 6.5, 7.0, 7.5])
t = rng.normal(5, 3, size=(10, 3, 4))
ds = xr.Dataset({"t2m": (("time", "lat", "lon"), t)},
                coords={"time": time, "lat": lat, "lon": lon})
```

In [ ]:
# Your solution here

:::{admonition} Solution
:class: note dropdown
```python
import numpy as np
import xarray as xr
rng = np.random.default_rng(0)
time = np.arange("2024-01-01", "2024-01-11", dtype="datetime64[D]")
lat = np.array([46.0, 46.5, 47.0]); lon = np.array([6.0, 6.5, 7.0, 7.5])
t = rng.normal(5, 3, size=(10, 3, 4))
ds = xr.Dataset({"t2m": (("time", "lat", "lon"), t)},
                coords={"time": time, "lat": lat, "lon": lon})
print(ds["t2m"].isel(time=0).mean().round(2).item())             # first day, spatial mean
print(ds["t2m"].sel(lat=47.0).mean(dim="time").round(2).values)  # one latitude, time mean
```
:::

## Exercise 6 — resample and a monthly climatology

Build a one-year daily temperature DataArray (construction below), compute its monthly means with `resample`, print how many there are, then use `groupby` to find the warmest calendar month (1–12).

```python
rng = np.random.default_rng(0)
time = np.arange("2024-01-01", "2025-01-01", dtype="datetime64[D]")
n = time.size; doy = np.arange(n)
t = 5 + -np.cos(2 * np.pi * doy / n) * 10 + rng.normal(0, 1.5, n)
da = xr.DataArray(t, dims="time", coords={"time": time}, attrs={"units": "degC"})
```

In [ ]:
# Your solution here

:::{admonition} Solution
:class: note dropdown
```python
import numpy as np
import xarray as xr
rng = np.random.default_rng(0)
time = np.arange("2024-01-01", "2025-01-01", dtype="datetime64[D]")
n = time.size; doy = np.arange(n)
t = 5 + -np.cos(2 * np.pi * doy / n) * 10 + rng.normal(0, 1.5, n)
da = xr.DataArray(t, dims="time", coords={"time": time}, attrs={"units": "degC"})
monthly = da.resample(time="MS").mean()
print(monthly["time"].size)
clim = da.groupby("time.month").mean()
print(int(clim.argmax("month").item()) + 1)
```
:::

## Exercise 7 — round-trip through netCDF

Create a 1D temperature DataArray named `t2m` with a `units` attribute, write it to `series.nc`, reopen it with `open_dataset`, and print the variable names and the recovered units.

In [ ]:
# Your solution here

:::{admonition} Solution
:class: note dropdown
```python
import numpy as np
import xarray as xr
da = xr.DataArray(np.arange(5.0), dims="time", name="t2m", attrs={"units": "degC"})
da.to_netcdf("series.nc")
reopened = xr.open_dataset("series.nc")
print(list(reopened.data_vars), reopened["t2m"].attrs["units"])
reopened.close()
```
:::